# AASIST + AST: Audio Spoof Detection with Audio Spectrogram Transformer (Colab with Resume Support)

This notebook trains the **AASIST-AST** hybrid model, combining AASIST with an Audio Spectrogram Transformer (AST) encoder. It includes **checkpointing and resume functionality** to overcome Google Colab's session limits.

## Architecture Overview
- **AASIST Branch**: Raw waveform → SincConv → ResNet encoder → Spectral & Temporal GATs → Graph pooling
- **AST Branch**: Raw waveform → Mel Spectrogram → Patch Embedding → Transformer Encoder → CLS token
- **Fusion**: Concatenation of both branches → Final binary classifier

## Expected Performance
The baseline AASIST achieves **EER: 0.83%, min t-DCF: 0.0275** on ASVspoof2019 LA eval set.  
The AASIST-AST hybrid is expected to improve upon this by leveraging global spectro-temporal attention from the Transformer.

---
**Runtime**: Use **GPU** (T4 or A100 recommended). Go to `Runtime → Change runtime type → GPU`.

## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB


## Step 2: Mount Google Drive for Checkpoints

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount أول مرة
drive.mount('/content/drive')

# مسار الحفظ
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/AASIST_AST_Checkpoints"
Path(DRIVE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

# فحص صلاحية الكتابة (مهم)
test_file = os.path.join(DRIVE_CHECKPOINT_DIR, ".write_test")
try:
    with open(test_file, "w") as f:
        f.write("ok")
    os.remove(test_file)
    print(f"✅ Google Drive mounted and writable.\nCheckpoint dir: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print(f"❌ Drive mounted but not writable: {e}")
    print("تأكد أنك وافقت على صلاحيات Google Drive بالكامل أثناء mount.")

Mounted at /content/drive
✅ Google Drive mounted and writable.
Checkpoint dir: /content/drive/MyDrive/AASIST_AST_Checkpoints


## Step 3: Clone Repository and Install Dependencies

In [ ]:
import os

# Change to /content/ directory to ensure correct cloning path
%cd /content/

# Clone the forked repository if it doesn't exist
if not os.path.exists("aasist"):
    !git clone https://github.com/ahmadSh96/aasist.git

# Change into the cloned repository directory
%cd aasist

# Switch to the AST integration branch
!git checkout feature/ast-integration
!git pull origin feature/ast-integration

# Install dependencies
!pip install -q torchcontrib soundfile torchaudio

print("\n✅ Setup complete!")

/content
Cloning into 'aasist'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 125 (delta 66), reused 67 (delta 34), pack-reused 10 (from 1)
Receiving objects: 100% (125/125), 1.46 MiB | 4.81 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/aasist
Branch 'feature/ast-integration' set up to track remote branch 'feature/ast-integration' from 'origin'.
Switched to a new branch 'feature/ast-integration'
From https://github.com/ahmadSh96/aasist
 * branch            feature/ast-integration -> FETCH_HEAD
Already up to date.
  Preparing metadata (setup.py) ... done

✅ Setup complete!


## Step 4: Download ASVspoof 2019 LA Dataset

> **Note**: The dataset is ~10GB. This will take several minutes.
> If you already have the dataset in your Drive, it will be copied to the local runtime.

In [ ]:
import os
from google.colab import drive

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define paths
drive_path = "/content/drive/MyDrive/LA.zip"
local_zip_path = "/content/LA.zip"
extract_path = "/content/aasist/LA"

# 2. Check if the extracted folder already exists
if not os.path.exists(extract_path):
    # 3. Check if the zip file exists in Drive
    if os.path.exists(drive_path):
        print("📦 Found LA.zip in Google Drive. Copying to local runtime...")
        !cp "{drive_path}" "{local_zip_path}"
    else:
        print("🌐 LA.zip not found in Drive. Downloading (~10GB)...")
        !wget -q --show-progress https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip -O "{local_zip_path}"

        # Save a copy to Drive for future use
        print("💾 Saving a copy to Google Drive for future use...")
        !cp "{local_zip_path}" "{drive_path}"

    # 4. Extract the zip file
    print("🔓 Extracting LA.zip...")
    !unzip -q "{local_zip_path}" -d /content/aasist

    # Remove local zip to save space
    !rm "{local_zip_path}"
    print("✅ Dataset ready!")
else:
    print("✅ Dataset folder 'LA' already exists in local runtime, skipping.")

📦 Found LA.zip in Google Drive. Copying to local runtime...
🔓 Extracting LA.zip...
✅ Dataset ready!


## Step 5: Verify Model Architecture

Let's inspect the AASIST-AST model to confirm it loads correctly and count parameters.

In [ ]:
import sys
sys.path.insert(0, "./")

import json
import torch

# Load config
with open("config/AASIST_AST.conf", "r") as f:
    config = json.load(f)

model_config = config["model_config"]

# Load model
from models.AASIST_AST import Model
model = Model(model_config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: AASIST-AST")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Test forward pass
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
dummy_input = torch.randn(2, 64600).to(device)  # batch=2, 4 seconds at 16kHz
with torch.no_grad():
    features, output = model(dummy_input)

print(f"\nForward pass test:")
print(f"  Input shape:    {dummy_input.shape}")
print(f"  Features shape: {features.shape}")
print(f"  Output shape:   {output.shape}")
print("\n✅ Model loaded and forward pass successful!")

/usr/local/lib/python3.12/dist-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (257) may be set too low.
  warnings.warn(


Model: AASIST-AST
Total parameters:     11,241,098
Trainable parameters: 11,241,098

Forward pass test:
  Input shape:    torch.Size([2, 64600])
  Features shape: torch.Size([2, 544])
  Output shape:   torch.Size([2, 2])

✅ Model loaded and forward pass successful!


## Step 6: Train the AASIST-AST Model (with Checkpointing)

In [ ]:
import os
import glob
import shutil

# Define output directory for this Colab session
COLAB_OUTPUT_DIR = "colab_exp_result"
os.makedirs(COLAB_OUTPUT_DIR, exist_ok=True)

print(f"DRIVE_CHECKPOINT_DIR: {DRIVE_CHECKPOINT_DIR}")
print("Listing contents of DRIVE_CHECKPOINT_DIR:")
!ls -R {DRIVE_CHECKPOINT_DIR}

# Find latest checkpoint in Google Drive
latest_checkpoint = None
checkpoint_files = sorted(glob.glob(f"{DRIVE_CHECKPOINT_DIR}/**/weights/checkpoint.pth", recursive=True), key=os.path.getmtime)
print(f"Found checkpoint files: {checkpoint_files}")
if checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    print(f"Found latest checkpoint: {latest_checkpoint}")

# Construct training command
train_command_parts = [
    "python main.py",
    "--config config/AASIST_AST.conf",
    f"--output_dir {COLAB_OUTPUT_DIR}",
    "--seed 1234",
    "--pretrained_aasist_path models/weights/AASIST.pth"
]

if latest_checkpoint:
    train_command_parts.append(f"--resume_checkpoint {latest_checkpoint}")

train_command = " ".join(train_command_parts)

print("Starting training with command:")
print(f"!{train_command}")
!{train_command}

# After training, copy results to Google Drive
print(f"\nCopying results from {COLAB_OUTPUT_DIR} to {DRIVE_CHECKPOINT_DIR}...")
model_tag_dirs = glob.glob(f"{COLAB_OUTPUT_DIR}/*_ep*_bs*")
if model_tag_dirs:
    actual_output_to_copy = model_tag_dirs[0]
    print(f"Copying actual output directory: {actual_output_to_copy} to {DRIVE_CHECKPOINT_DIR}/")
    !cp -r {actual_output_to_copy}/* {DRIVE_CHECKPOINT_DIR}/
    print("✅ Results copied to Google Drive!")
else:
    print("❌ No output directory found to copy.")

## Step 7: Evaluate Results

In [ ]:
import os
import glob

print("--- Final Evaluation Results ---")
log_files = glob.glob(f"{DRIVE_CHECKPOINT_DIR}/metric_log.txt")
if log_files:
    with open(log_files[0], "r") as f:
        print(f.read())
else:
    print("No metric log found yet. Training might still be in progress or failed.")

print("\n--- EER and t-DCF Summary ---")
summary_files = glob.glob(f"{DRIVE_CHECKPOINT_DIR}/t-DCF_EER.txt")
if summary_files:
    with open(summary_files[0], "r") as f:
        print(f.read())
else:
    print("No final summary found yet.")

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import shutil

# 1) اعمل remount إجباري حتى نتجنب mount قديم read-only
drive.mount('/content/drive', force_remount=True)

# 2) مسار الحفظ على درايف
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/AASIST_AST_Checkpoints"
Path(DRIVE_CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

# 3) اختبار كتابة فعلي (هذا أهم جزء)
test_file = os.path.join(DRIVE_CHECKPOINT_DIR, ".colab_write_test")
try:
    with open(test_file, "w") as f:
        f.write("write_ok")
    os.remove(test_file)
    print(f"✅ Google Drive mounted with WRITE access.\nCheckpoint dir: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print("❌ Drive mounted but NOT writable.")
    print("سبب الخطأ:", repr(e))
    raise RuntimeError(
        "Google Drive is read-only or permission denied. "
        "افتح mount من جديد واختر الحساب الصحيح ووافق على الصلاحيات."
    )

Mounted at /content/drive
✅ Google Drive mounted with WRITE access.
Checkpoint dir: /content/drive/MyDrive/AASIST_AST_Checkpoints
